In [181]:
import yaml
import os

with open("config/config.yml", "r") as file:
    config = yaml.safe_load(file)




In [182]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]

In [183]:

# Csv Files
csv_file= config['file']['csv']

# Temp Files
    # Bronze Temp Files
deduplicate_temp= config['temp']['bronze']['deduplicate']

    # Silver Temp Files
silver_nonull_temp= config['temp']['silver']['nonull']
silver_concat_temp= config['temp']['silver']['concat']


# Databases
bronze_db= config['database']['bronze']
silver_db= config['database']['silver']

# Delta Tables
    # Bronze delta table
bronze_original= config['delta']['bronze']['original_data']
bronze_final= config['delta']['bronze']['final_data']

    # Silver delta table
silver_original= config['delta']['silver']['original_data']
silver_transformed= config['delta']['silver']['transformed_data']
silver_final= config['delta']['silver']['final_data']

# Null Removel Comulns Array

nn_items= config['null_removal']
print(nn_items)

# dimensions
dimensions = config['dimension']

for table,columns in dimensions.items():
    print(table,columns)


# Facts
facts= config['facts']
print(facts)

['product_id', 'category_id', 'category_code', 'brand']
dim_product ['product_id', 'category_id', 'category_code', 'brand']
dim_event_type ['event_type']
dim_user ['user_id']
['event_time', 'event_type', 'product_id', 'price', 'user_id', 'user_session']


In [16]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DeltaLakeLocal")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
df=spark.read.csv(csv_file,inferSchema=True,header=True)

df.createOrReplaceTempView("bronze_original_temp")

In [14]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [6]:
print("Creating the bronze database")
spark.sql(f"""
        create database if not exists bronze_db
""")



Creating the bronze database


DataFrame[]

In [7]:
spark.sql(f"""
    create or replace table {bronze_db}.{bronze_original}
    using delta 
    as 
    select * from bronze_original_temp
""")

DataFrame[]

In [13]:
print("Deleting exact duplicate rows from the data")
spark.sql(f"""
    create or replace temp view {deduplicate_temp}
    as select distinct * from {bronze_db}.{bronze_original}
""")

print("creating new delta table bronze_delta as the final bronze layer table")

spark.sql(f"""
        create or replace table {bronze_db}.{bronze_final}
        using delta
        as
        select * from {deduplicate_temp}
""")

print("Success")



Deleting exact duplicate rows from the data
creating new delta table bronze_delta as the final bronze layer table
Success


In [ ]:
print("Completiion of Bronze Layer")
print("Start of siver layer")

In [36]:
spark.sql(f"""
        create database if not exists {silver_db}
""")

DataFrame[]

In [96]:
spark.sql(f"""
    create or replace table {silver_db}.{silver_original}
    using delta
    as
    select * from  {bronze_db}.{bronze_final}
""")

DataFrame[]

In [97]:
spark.sql(f"""
    select event_type from {silver_db}.{silver_original}
    where event_type is null
""").show()

+----------+
|event_type|
+----------+
+----------+



In [98]:
print("Starting of No nullification of the data ")

string= f""""""
columns= ",".join(nn_items)

for item in nn_items:

    if not string:
        string += f"""COALESCE({item},"Unknown") as {item}"""
    else:
        string += f""",COALESCE({item},"Unknown") as {item}"""
    
print(string)
print(columns)

spark.sql(f"""
        create or replace temp view {silver_nonull_temp}
        as select * Except({columns}),
        {string}
        from {silver_db}.{silver_original}
""")


Starting of No nullification of the data 
COALESCE(product_id,"Unknown") as product_id,COALESCE(category_id,"Unknown") as category_id,COALESCE(category_code,"Unknown") as category_code,COALESCE(brand,"Unknown") as brand
product_id,category_id,category_code,brand


DataFrame[]

In [100]:
print(f"Removed null values from [ {columns} ]")

Removed null values from [ product_id,category_id,category_code,brand ]


In [101]:
print("Concatting event_time + user_id into usersession")


spark.sql(f"""
    create or replace temp view {silver_concat_temp}
    as 
    select * except(user_session),
    coalesce(user_session, Concat(event_time,"_",user_id)) as user_session
    from {silver_nonull_temp}
""")

print("Success!!")

Concatting event_time + user_id into usersession
Success!!


In [105]:
spark.sql(f"select * from silver_db.silver_original").show(10)

+-------------------+----------+----------+-------------------+--------------------+---------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|    brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+---------+------+---------+--------------------+
|2019-11-09 17:27:17|  purchase|     24342|1487580013053083824| stationery.cartrige|depilflax|  3.02|569303969|4ba76297-29cf-440...|
|2019-11-09 20:58:03|      cart|   5749199|1487580013053083824| stationery.cartrige|  italwax|  1.98|563916825|5abf17cd-2f4e-4f5...|
|2019-11-09 22:54:44|      view|   5771614|2193074740619379535|furniture.living_...| kosmekka|150.79|569417715|5ad1a76b-3766-4c6...|
|2019-11-10 12:16:04|      cart|   5749198|1487580013053083824| stationery.cartrige|  italwax|  1.98|463833092|31f3176b-eb20-440...|
|2019-11-10 12:36:50|      view|   5889696|2007399943458784057|      

In [204]:
print("""Starting of dimension modeling """)
print("Creating a Delta Table with all final transformed data")

spark.sql(f"""
    create or replace table {silver_db}.{silver_transformed}
    using delta
    as
    select * from {silver_concat_temp}
""")
print("Succesfully created silver_transformed table!!")

print("Starting to create Delta Dimension Tables according to the configs")




Starting of dimension modeling 
Creating a Delta Table with all final transformed data
Succesfully created silver_transformed table!!
Starting to create Delta Dimension Tables according to the configs


In [203]:
spark.sql(f"drop table silver_db.silver_transformed")

DataFrame[]

In [ ]:
# spark.sql(f"drop table silver_db.dim_product")

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `spark_catalog`.`silver_db`.`dim_product` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01

In [168]:
spark.sql("select * from silver_db.dim_product order by  product_id desc").show()

+-------------+----------+-------------------+-------------+-------+
|product_id_id|product_id|        category_id|category_code|  brand|
+-------------+----------+-------------------+-------------+-------+
|        44019|   5909246|2187790129827939246|      Unknown|  dewal|
|        44020|   5909246|2187790129827939246|      Unknown|Unknown|
|        44018|   5909245|2187790129827939246|      Unknown|  dewal|
|        44016|   5909244|2187790129827939246|      Unknown|  dewal|
|        44017|   5909244|2187790129827939246|      Unknown|Unknown|
|        44014|   5909242|2187790129827939246|      Unknown|  dewal|
|        44015|   5909242|2187790129827939246|      Unknown|Unknown|
|        44012|   5909241|2187790129827939246|      Unknown|Unknown|
|        44013|   5909241|2187790129827939246|      Unknown|  dewal|
|        44010|   5909240|2187790129827939246|      Unknown|Unknown|
|        44011|   5909240|2187790129827939246|      Unknown|  dewal|
|        44009|   5909239|21877901

In [206]:

for table_name,columns in dimensions.items():
    table= table_name
    columns_string= ",".join(columns)
    fact_match_array= []

    for column in columns:
        fact_match_array.append(f"st.{column}=tn.{column}")

    match_condition= " and ".join(fact_match_array)

    print(match_condition)

    print(f"Creating dimension table for {table} \n ")

    spark.sql(f"""
        create or replace table {silver_db}.{table}
        using delta
        as
        select 
        Row_Number() over (order by {columns[0]}) as {columns[0]+"_id"},
        {columns_string} from
        (
        select distinct
        {columns_string}
        from {silver_db}.{silver_transformed}
        )
""")
    
    print("Success \n")

    print(f"Updating fact table for {table} \n")

    spark.sql(f"""
    merge into {silver_db}.{silver_transformed} st
    using {silver_db}.{table_name} tn
    on {match_condition}
    when Matched Then
    update set st.{columns[0]} = tn.{columns[0]+"_id"}
    
""")

    print(f"st.{columns[0]} = tn.{columns[0]+"_id"}")
    print(f"Success \n")

st.product_id=tn.product_id and st.category_id=tn.category_id and st.category_code=tn.category_code and st.brand=tn.brand
Creating dimension table for dim_product 
 
Success 

Updating fact table for dim_product 

st.product_id = tn.product_id_id
Success 

st.event_type=tn.event_type
Creating dimension table for dim_event_type 
 
Success 

Updating fact table for dim_event_type 

st.event_type = tn.event_type_id
Success 

st.user_id=tn.user_id
Creating dimension table for dim_user 
 
Success 

Updating fact table for dim_user 

st.user_id = tn.user_id_id
Success 



In [208]:
spark.sql("select * from silver_db.silver_transformed").show()

+-------------------+----------+------+-------+----------+-------------------+--------------------+---------+--------------------+
|         event_time|event_type| price|user_id|product_id|        category_id|       category_code|    brand|        user_session|
+-------------------+----------+------+-------+----------+-------------------+--------------------+---------+--------------------+
|2019-11-09 14:26:47|         4|  9.21| 213526|     41826|2007399943458784057|       apparel.glove|   benovy|d373d212-b866-428...|
|2019-11-09 14:29:53|         4|150.79| 213971|     12862|2193074740619379535|furniture.living_...| kosmekka|284f8088-f6c9-4bf...|
|2019-11-09 14:46:39|         1|  3.02| 213410|      1074|1487580013053083824| stationery.cartrige|depilflax|886943a8-fa9d-468...|
|2019-11-09 17:18:20|         1|  2.14| 134709|     16907|1487580013053083824| stationery.cartrige|  italwax|cad2b4b3-df70-445...|
|2019-11-09 17:35:50|         4| 55.56| 215060|     33756|1487580006350586771|appli

In [ ]:
print("Creating fact table for the silver layer")
facts=",".join(facts)
print(facts)

spark.sql(f"""create or replace table {silver_db}.{silver_final}
              using delta
              as
              select {facts}
              from {silver_db}.{silver_transformed}
    """)

print("Success!!")

spark.sql(f"select * from {silver_db}.{silver_final}").show()

Creating fact table for the silver layer


ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near ','. SQLSTATE: 42601 (line 4, pos 41)

== SQL ==
create or replace table silver_db.silver_final
              using delta
              as
              select e,v,e,n,t,_,t,i,m,e,,,e,v,e,n,t,_,t,y,p,e,,,p,r,o,d,u,c,t,_,i,d,,,p,r,i,c,e,,,u,s,e,r,_,i,d,,,u,s,e,r,_,s,e,s,s,i,o,n
-----------------------------------------^^^
              from silver_db.silver_transformed
    
